# Efficient AQNG vs PennyLane QNG — Iris / Colab

This notebook benchmarks the efficient AQNG implementation from:

`AHDMarwan/aqng`

Main AQNG efficiency settings:

\[
B_{\rm loss}=10,\qquad B_{\rm metric}=2,\qquad K_{\rm metric}=4.
\]

The accessible Fisher geometry is refreshed every four optimization steps and is estimated on a smaller metric mini-batch. On `default.qubit`, expectation-value gradients use PennyLane's adjoint differentiation.

The PennyLane-QNG baseline uses the same model, initialization, loss batches and number of optimization steps.

In [ ]:
!pip -q install "pennylane>=0.45,<0.46" scikit-learn pandas matplotlib
!pip -q install --upgrade "git+https://github.com/AHDMarwan/aqng.git"

import time
import pennylane as qml
from pennylane import numpy as np
import numpy as onp
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from aqng_efficient import AQNGEfficientOptimizer

print("PennyLane:", qml.__version__)

## 1. Real dataset

In [ ]:
SEED = 7
rng = onp.random.default_rng(SEED)

iris = load_iris()
mask = iris.target < 2
X = iris.data[mask].astype(float)
y = iris.target[mask].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = onp.pi * onp.tanh(X_train / 2.0)
X_test = onp.pi * onp.tanh(X_test / 2.0)

y_train_pm = 2.0 * y_train - 1.0
y_test_pm = 2.0 * y_test - 1.0

print("train:", X_train.shape, "test:", X_test.shape)

## 2. VQC and accessible readout

AQNG uses all one- and two-body diagonal \(Z\)-strings. For four qubits this gives \(r=10\).

`adjoint` is used only for the simulator benchmark. For finite-shot hardware, switch the differentiable QNodes to `parameter-shift`.

In [ ]:
N_QUBITS = 4
N_LAYERS = 2
dev = qml.device("default.qubit", wires=N_QUBITS)

z_terms = [(i,) for i in range(N_QUBITS)]
z_terms += [(i, j) for i in range(N_QUBITS) for j in range(i + 1, N_QUBITS)]

def ansatz(theta, x):
    qml.AngleEmbedding(x, wires=range(N_QUBITS), rotation="Y")
    for l in range(N_LAYERS):
        for w in range(N_QUBITS):
            qml.Rot(*theta[l, w], wires=w)
        for w in range(N_QUBITS):
            qml.CNOT(wires=[w, (w + 1) % N_QUBITS])

@qml.qnode(dev, interface="autograd", diff_method="adjoint")
def pred_qnode(theta, x):
    ansatz(theta, x)
    return qml.expval(qml.PauliZ(0))

@qml.qnode(dev, interface="autograd", diff_method="adjoint")
def feature_qnode(theta, x):
    ansatz(theta, x)
    obs = []
    for term in z_terms:
        op = qml.PauliZ(term[0])
        for w in term[1:]:
            op = op @ qml.PauliZ(w)
        obs.append(qml.expval(op))
    return tuple(obs)

# Covariance is not differentiated, so exact probabilities are sufficient
# for this simulator benchmark.
@qml.qnode(dev, interface="autograd")
def prob_qnode(theta, x):
    ansatz(theta, x)
    return qml.probs(wires=range(N_QUBITS))

basis = onp.arange(2**N_QUBITS)
bits = ((basis[:, None] >> onp.arange(N_QUBITS - 1, -1, -1)) & 1)
zvals = 1.0 - 2.0 * bits
SIGN = onp.stack(
    [onp.prod(zvals[:, list(term)], axis=1) for term in z_terms],
    axis=1,
)

P = N_LAYERS * N_QUBITS * 3
print("parameters p =", P)
print("readout features r =", len(z_terms))

## 3. Loss and metric closures

In [ ]:
def make_cost(Xb, yb):
    Xb = onp.asarray(Xb)
    yb = onp.asarray(yb)

    def cost(theta):
        pred = qml.math.stack([pred_qnode(theta, x) for x in Xb])
        return qml.math.mean((pred - yb) ** 2)

    return cost

def make_metric_fns(Xm):
    Xm = onp.asarray(Xm)

    def features(theta):
        return qml.math.stack(
            [qml.math.stack(feature_qnode(theta, x)) for x in Xm]
        )

    def covariance(theta):
        covs = []
        for x in Xm:
            p = onp.asarray(qml.math.toarray(prob_qnode(theta, x)), dtype=float)
            mean = p @ SIGN
            second = SIGN.T @ (p[:, None] * SIGN)
            covs.append(second - onp.outer(mean, mean))
        return onp.stack(covs)

    return features, covariance

def full_loss(theta, X, ypm):
    vals = qml.math.stack([pred_qnode(theta, x) for x in X])
    return float(qml.math.mean((vals - ypm) ** 2))

def accuracy(theta, X, y):
    pred = onp.array([float(pred_qnode(theta, x)) for x in X])
    return float(onp.mean((pred >= 0.0).astype(int) == y))

## 4. PennyLane QNG batch metric

In [ ]:
single_qng_metric = qml.metric_tensor(pred_qnode, approx="block-diag")

def make_qng_metric(Xb):
    Xb = onp.asarray(Xb)

    def metric(theta):
        mats = []
        p = int(onp.prod(theta.shape))
        for x in Xb:
            g = single_qng_metric(theta, x)
            mats.append(qml.math.reshape(g, (p, p)))
        return qml.math.mean(qml.math.stack(mats), axis=0)

    return metric

## 5. Common protocol

In [ ]:
STEPS = 20
LOSS_BATCH = 10
METRIC_BATCH = 2
METRIC_EVERY = 4

AQNG_LR = 0.03
QNG_LR = 0.03
LAM = 1e-3
COV_LAM = 1e-3

theta0 = np.array(
    rng.normal(scale=0.15, size=(N_LAYERS, N_QUBITS, 3)),
    requires_grad=True,
)

batch_ids = [
    rng.choice(len(X_train), size=LOSS_BATCH, replace=False)
    for _ in range(STEPS)
]

## 6. Efficient AQNG

In [ ]:
theta_a = np.array(theta0, requires_grad=True)

aqng = AQNGEfficientOptimizer(
    stepsize=AQNG_LR,
    lam=LAM,
    cov_lam=COV_LAM,
    metric_every=METRIC_EVERY,
    solver="auto",
    rcond=1e-8,
)

hist_a = []
t0 = time.perf_counter()

for step, ids in enumerate(batch_ids):
    loss_ids = ids
    metric_ids = ids[:METRIC_BATCH]

    cost = make_cost(X_train[loss_ids], y_train_pm[loss_ids])
    features, covariance = make_metric_fns(X_train[metric_ids])

    theta_a, old_cost = aqng.step_and_cost(
        cost,
        theta_a,
        feature_fn=features,
        covariance_fn=covariance,
    )

    d = aqng.diagnostics
    row = {
        "optimizer": "AQNG-efficient",
        "step": step + 1,
        "batch_loss_before": float(old_cost),
        "train_loss": full_loss(theta_a, X_train, y_train_pm),
        "test_loss": full_loss(theta_a, X_test, y_test_pm),
        "test_acc": accuracy(theta_a, X_test, y_test),
        "elapsed_s": time.perf_counter() - t0,
        "metric_recomputed": d.metric_recomputed,
        "metric_age": d.metric_age,
        "metric_batch": d.batch_size,
        "feature_dim": d.feature_dim,
        "metric_rank": d.metric_rank,
        "metric_condition": d.metric_condition,
        "solver": d.solver,
        "solve_dimension": d.solve_dimension,
        "gradient_s": d.gradient_seconds,
        "metric_s": d.metric_seconds,
        "solve_s": d.solve_seconds,
        "optimizer_step_s": d.total_step_seconds,
    }
    hist_a.append(row)

    print(
        f"AQNG {step+1:02d} | test={row['test_loss']:.4f} "
        f"acc={row['test_acc']:.3f} | refresh={d.metric_recomputed} "
        f"metric={d.metric_seconds:.3f}s solve={d.solve_seconds:.4f}s"
    )

## 7. PennyLane QNG baseline

In [ ]:
theta_q = np.array(theta0, requires_grad=True)
qng = qml.QNGOptimizer(stepsize=QNG_LR, approx=None, lam=LAM)

hist_q = []
t0 = time.perf_counter()

for step, ids in enumerate(batch_ids):
    cost = make_cost(X_train[ids], y_train_pm[ids])
    metric_fn = make_qng_metric(X_train[ids])

    theta_q, old_cost = qng.step_and_cost(
        cost,
        theta_q,
        metric_tensor_fn=metric_fn,
    )

    row = {
        "optimizer": "PennyLane-QNG",
        "step": step + 1,
        "batch_loss_before": float(old_cost),
        "train_loss": full_loss(theta_q, X_train, y_train_pm),
        "test_loss": full_loss(theta_q, X_test, y_test_pm),
        "test_acc": accuracy(theta_q, X_test, y_test),
        "elapsed_s": time.perf_counter() - t0,
    }
    hist_q.append(row)

    print(
        f"QNG  {step+1:02d} | test={row['test_loss']:.4f} "
        f"acc={row['test_acc']:.3f}"
    )

## 8. Compare and save

In [ ]:
df = pd.DataFrame(hist_a + hist_q)

summary = (
    df.sort_values("step")
      .groupby("optimizer")
      .tail(1)[["optimizer", "train_loss", "test_loss", "test_acc", "elapsed_s"]]
      .reset_index(drop=True)
)

display(summary)

aqng_only = df[df["optimizer"] == "AQNG-efficient"].copy()
display(
    aqng_only[
        [
            "step", "metric_recomputed", "metric_age",
            "gradient_s", "metric_s", "solve_s",
            "solver", "solve_dimension"
        ]
    ]
)

plt.figure(figsize=(7, 4))
for name, g in df.groupby("optimizer"):
    plt.plot(g["step"], g["test_loss"], marker="o", label=name)
plt.yscale("log")
plt.xlabel("Optimization step")
plt.ylabel("Test MSE")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

plt.figure(figsize=(7, 4))
for name, g in df.groupby("optimizer"):
    plt.plot(g["elapsed_s"], g["test_loss"], marker="o", label=name)
plt.yscale("log")
plt.xlabel("Wall-clock seconds")
plt.ylabel("Test MSE")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

df.to_csv("/content/aqng_efficient_vs_qng_history.csv", index=False)
summary.to_csv("/content/aqng_efficient_vs_qng_summary.csv", index=False)

print("saved /content/aqng_efficient_vs_qng_history.csv")
print("saved /content/aqng_efficient_vs_qng_summary.csv")

## 9. Hardware switch

For finite-shot hardware:

1. change differentiable feature/prediction QNodes to `diff_method="parameter-shift"`;
2. estimate readout covariance from computational-basis bitstrings with:

```python
from aqng_pennylane import z_covariance_from_bitstrings
```

All diagonal \(Z\)-string covariance entries can be estimated from the same bitstrings.

For publication-level benchmarking, compare loss/accuracy against optimization steps, wall-clock time, circuit executions, and total shots.